# nanoql: first steps

nanoql prices an instrument as a plain function of its inputs: the instrument, an evaluation context that carries the date, and the market data its engine takes. This notebook runs in Google Colab, or locally after `pip install nanoql jupyter matplotlib`. Its option cell follows QuantLib's EquityOption example, and its SABR cell takes the market data of QuantLib's swaption cube tests.

The first cell installs the library, and matplotlib for the charts.

In [ ]:
%pip install "nanoql>=2.0,<3" matplotlib

## A first swap

A five-year SOFR swap on a flat curve. The curve, the swap and the evaluation date are values, and the engine is a function of them.

In [ ]:
import datetime

from nanoql import ConventionSet, DiscountingSwapEngine, EvaluationContext, FlatForward, MakeOIS, Period, TimeUnit

today = datetime.date(2026, 10, 15)
context = EvaluationContext(today)
qs = ConventionSet.quant_lib()
curve = FlatForward(today, 0.036, qs.day_counter("Actual365Fixed"))

swap = (MakeOIS(Period(5, TimeUnit.Years), qs.overnight_index("Sofr", curve))
        .with_fixed_rate(0.036).with_nominal(10e6)
        .to_overnight_indexed_swap(context))

result = DiscountingSwapEngine.calculate(swap, context, curve)
print(f"NPV {result.npv:,.2f}, fair rate {result.fair_rate:.4%}")

## A SOFR curve from swap quotes

A curve bootstrapped from six OIS quotes, its discount factors at the quotes' tenors:

In [ ]:
from nanoql import OISRateHelper, PiecewiseLogLinearDiscount

quotes = {1: 0.0372, 2: 0.0358, 3: 0.0352, 5: 0.0350, 7: 0.0353, 10: 0.0360}
sofr = qs.overnight_index("Sofr")
helpers = [OISRateHelper(2, Period(years, TimeUnit.Years), rate, sofr, context,
                         forward_start=Period(0, TimeUnit.Days))  # spot-starting swaps
           for years, rate in quotes.items()]
sofr_curve = PiecewiseLogLinearDiscount(today, helpers, qs.day_counter("Actual365Fixed"), context)

for years, rate in quotes.items():
    print(f"{years:>2}Y  discount {sofr_curve.discount(float(years)):.6f}  quote {rate:.4%}")

The same swap priced on the bootstrapped curve:

In [ ]:
swap = (MakeOIS(Period(5, TimeUnit.Years), qs.overnight_index("Sofr", sofr_curve))
        .with_fixed_rate(0.036).with_nominal(10e6)
        .to_overnight_indexed_swap(context))
result = DiscountingSwapEngine.calculate(swap, context, sofr_curve)
print(f"NPV {result.npv:,.2f}, fair rate {result.fair_rate:.4%}")

## Two curves: €STR discounting, Euribor forecasting

A swap's floating leg is forecast on its index's curve and discounted on the overnight curve. Here €STR is bootstrapped from OIS quotes, then Euribor 6M from its fixing and swaps against it, priced with €STR discounting: the structure of QuantLib's MulticurveBootstrapping example, on this notebook's own quotes. Every input of the Euribor 6M curve is on the 6M index. Both curves interpolate the discounts' logarithms by a monotonic cubic spline.

In [ ]:
from nanoql import (CubicInterpolationDerivativeApprox, DepositRateHelper, LogCubic, PiecewiseLogCubicDiscount,
                    SwapRateHelper)

target = qs.calendar("TARGET")
act365 = qs.day_counter("Actual365Fixed")
monotonic = LogCubic(CubicInterpolationDerivativeApprox.Spline, True)  # QuantLib's MonotonicLogCubic

# €STR, from overnight indexed swaps.
estr_quotes = {1: 0.0195, 2: 0.0201, 3: 0.0208, 5: 0.0222, 7: 0.0235, 10: 0.0251}
estr_curve = PiecewiseLogCubicDiscount(
    today,
    [OISRateHelper(2, Period(years, TimeUnit.Years), rate, qs.overnight_index("Estr"), context,
                   forward_start=Period(0, TimeUnit.Days))  # spot-starting swaps
     for years, rate in estr_quotes.items()],
    act365, context, i=monotonic)

# Euribor 6M, from its fixing and swaps against it, the swaps discounted on €STR. Each input is on the 6M index:
# a deposit of another tenor belongs on that tenor's curve.
euribor6m = qs.ibor_index("Euribor6M")
fixing_quote = 0.0212  # today's Euribor 6M fixing, as a deposit on the index
swap_quotes = {2: 0.0228, 3: 0.0234, 5: 0.0247, 7: 0.0259, 10: 0.0273}
euribor_helpers = [DepositRateHelper(fixing_quote, euribor6m, context)]
euribor_helpers += [SwapRateHelper(rate, qs.swap_index("EuriborSwapIsdaFixA", Period(years, TimeUnit.Years)), context,
                                   fwd_start=Period(0, TimeUnit.Days), discounting_curve=estr_curve)
                    for years, rate in swap_quotes.items()]
euribor_curve = PiecewiseLogCubicDiscount(today, euribor_helpers, act365, context, i=monotonic)

for years in (1, 2, 5, 10):
    print(f"{years:>2}Y  €STR discount {estr_curve.discount(float(years)):.6f}  "
          f"Euribor 6M discount {euribor_curve.discount(float(years)):.6f}")

Each curve drawn in its quotes' own terms, by tenor: €STR's par rate of a spot-starting OIS, and Euribor 6M's fixing at six months, then the par rate of a swap against it discounted on €STR. Each quote lies on its line, the bootstrap's condition:

In [ ]:
import matplotlib.pyplot as plt

from nanoql import MakeVanillaSwap

estr_on_curve = qs.overnight_index("Estr", estr_curve)
euribor_on_curve = qs.ibor_index("Euribor6M", euribor_curve)


def ois_rate(months):
    """The €STR curve's par OIS rate for a spot-starting swap of the given months."""
    swap = MakeOIS(Period(months, TimeUnit.Months), estr_on_curve).to_overnight_indexed_swap(context)
    return DiscountingSwapEngine.calculate(swap, context, estr_curve).fair_rate


def swap_rate(months):
    """The par rate of a spot-starting swap against Euribor 6M, forecast on its curve and discounted on €STR."""
    swap = MakeVanillaSwap(Period(months, TimeUnit.Months), euribor_on_curve).to_vanilla_swap(context)
    return DiscountingSwapEngine.calculate(swap, context, estr_curve).fair_rate


ois_months = range(6, 121, 3)
swap_months = range(12, 121, 3)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot([m / 12 for m in ois_months], [100 * ois_rate(m) for m in ois_months], label="€STR OIS par rate")
ax.scatter(list(estr_quotes), [100 * r for r in estr_quotes.values()], zorder=3, label="€STR OIS quotes")
# The 6M curve starts at its 6M fixing, the curve's own forecast of today's fixing, then the swaps' par rates.
ax.plot([0.5] + [m / 12 for m in swap_months],
        [100 * euribor_on_curve.fixing(today, context, True)] + [100 * swap_rate(m) for m in swap_months],
        label="Euribor 6M: the fixing, then swap par rates")
ax.scatter([0.5, *swap_quotes], [100 * fixing_quote, *(100 * r for r in swap_quotes.values())], marker="s",
           zorder=3, label="Euribor 6M fixing and swap quotes")
ax.set_xlabel("Years")
ax.set_ylabel("Rate (%)")
ax.set_title("Each curve's rates in its quotes' own terms, and the quotes")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncols=2)
plt.show()

A five-year swap forecast on Euribor 6M and discounted on €STR, and the same swap a year forward. The spot one's fair rate is its quote:

In [ ]:
for label, start in (("5Y swap", Period(0, TimeUnit.Days)), ("1Y-forward 5Y swap", Period(1, TimeUnit.Years))):
    swap = (MakeVanillaSwap(Period(5, TimeUnit.Years), qs.ibor_index("Euribor6M", euribor_curve))
            .with_fixed_rate(0.025).with_forward_start(start).with_nominal(1e6)
            .to_vanilla_swap(context))
    result = DiscountingSwapEngine.calculate(swap, context, estr_curve)  # forecast on Euribor, discounted on €STR
    print(f"{label:>18}: NPV {result.npv:>10,.2f}, fair rate {result.fair_rate:.4%}")

## A SABR smile

The 1Y into 10Y swaption smile of QuantLib's swaption cube tests' market data, 14.70% at the money and the market's spreads at four strikes, on a flat 5% curve, fitted by SABR with β fixed at 0.5. The fitted parameters make a smile section, the smile an engine reads at any strike:

In [ ]:
from nanoql import BusinessDayConvention, SABRInterpolation, SabrSmileSection

flat = FlatForward(today, 0.05, act365)
swap_index = qs.swap_index("EuriborSwapIsdaFixA", Period(10, TimeUnit.Years), flat)
option_date = target.advance(today, Period(1, TimeUnit.Years), BusinessDayConvention.ModifiedFollowing)
expiry = act365.year_fraction(today, option_date)
forward = swap_index.fixing(option_date, context)

# The 1Y into 10Y smile: 14.70% at the money, and the market's spreads to it at four strikes about the forward.
strikes = [forward + s for s in (-0.020, -0.005, 0.0, 0.005, 0.020)]
vols = [0.1470 + s for s in (0.0729, 0.0086, 0.0, -0.0024, 0.0098)]
sabr = SABRInterpolation(strikes, vols, expiry, forward, alpha=None, beta=0.5, nu=None, rho=None,
                         alpha_is_fixed=False, beta_is_fixed=True, nu_is_fixed=False, rho_is_fixed=False)
smile = SabrSmileSection(expiry, forward, [sabr.alpha, sabr.beta, sabr.nu, sabr.rho])

print(f"forward {forward:.4%}; alpha {sabr.alpha:.6f}, beta {sabr.beta:.1f} (fixed), nu {sabr.nu:.6f}, "
      f"rho {sabr.rho:.6f}")
for strike, vol in zip(strikes, vols):
    print(f"strike {strike:.4%}  quote {vol:.2%}  SABR {smile.volatility(strike):.4%}")

The smile across strikes, with the five quotes marked:

In [ ]:
grid = [forward + bp * 1e-4 for bp in range(-250, 251, 5)]
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot([100 * k for k in grid], [100 * smile.volatility(k) for k in grid], label="SABR smile")
ax.scatter([100 * k for k in strikes], [100 * v for v in vols], color="black", zorder=3, label="quotes")
ax.axvline(100 * forward, color="grey", linestyle=":", label="forward")
ax.set_xlabel("Strike (%)")
ax.set_ylabel("Black volatility (%)")
ax.set_title("1Y into 10Y: the SABR fit and its quotes")
ax.legend()
plt.show()

## An American put

QuantLib's EquityOption example's put, struck at 40 on a spot of 36, with a 6% rate, no dividends, 20% volatility and a year to expiry: European by Black-Scholes, and American by Barone-Adesi and Whaley's approximation.

In [ ]:
from nanoql import (AmericanExercise, AnalyticEuropeanEngine, BaroneAdesiWhaleyApproximationEngine, BlackConstantVol,
                    BlackScholesMertonProcess, EuropeanExercise, OptionType, PlainVanillaPayoff, VanillaOption)

trade_date = datetime.date(1998, 5, 15)
settlement = datetime.date(1998, 5, 17)
maturity = datetime.date(1999, 5, 17)
option_context = EvaluationContext(trade_date)
act365 = qs.day_counter("Actual365Fixed")
process = BlackScholesMertonProcess(
    36.0,
    FlatForward(settlement, 0.0, act365),
    FlatForward(settlement, 0.06, act365),
    BlackConstantVol(settlement, qs.calendar("TARGET"), 0.20, act365))

payoff = PlainVanillaPayoff(OptionType.Put, 40.0)
european = VanillaOption(payoff, EuropeanExercise(maturity))
american = VanillaOption(payoff, AmericanExercise(settlement, maturity))

print(f"European, Black-Scholes:       {AnalyticEuropeanEngine.calculate(european, option_context, process).npv:.6f}")
print(f"American, Barone-Adesi-Whaley: "
      f"{BaroneAdesiWhaleyApproximationEngine.calculate(american, option_context, process).npv:.6f}")

## A default curve from CDS spreads

A hazard-rate curve bootstrapped from one-, three- and five-year CDS spreads, and its survival probabilities:

In [ ]:
from nanoql import (BusinessDayConvention, CreditDefaultSwap, DateGenerationRule, Frequency, MakeSchedule,
                    MidPointCdsEngine, PiecewiseFlatHazardRate, ProtectionSide, SpreadCdsHelper)

target = qs.calendar("TARGET")
act360 = qs.day_counter("Actual360")
recovery = 0.4
spreads = {1: 0.0080, 3: 0.0110, 5: 0.0135}
cds_helpers = [
    SpreadCdsHelper(spread, Period(years, TimeUnit.Years), 0, target, Frequency.Quarterly,
                    BusinessDayConvention.Following, DateGenerationRule.TwentiethIMM, act360, recovery, curve, context)
    for years, spread in spreads.items()]
hazard_curve = PiecewiseFlatHazardRate(today, cds_helpers, act365, context)

for years in spreads:
    print(f"{years}Y survival probability {hazard_curve.survival_probability(float(years)):.4%}")

A five-year CDS at the quoted spread, priced on that curve:

In [ ]:
maturity = target.adjust(today.replace(year=today.year + 5), BusinessDayConvention.Following)
schedule = (MakeSchedule().from_(today).to(maturity)
            .with_frequency(Frequency.Quarterly).with_calendar(target)
            .with_termination_date_convention(BusinessDayConvention.Unadjusted)
            .with_rule(DateGenerationRule.TwentiethIMM)
            .to_schedule())
cds = CreditDefaultSwap(ProtectionSide.Buyer, 10e6, 0.0135, schedule, BusinessDayConvention.Following, act360)
cds_result = MidPointCdsEngine.calculate(cds, context, hazard_curve, recovery, curve)
# The NPV is round-off, its sign the platform's last bits; rounded and added to 0.0, it prints as 0.00.
print(f"NPV {round(cds_result.npv, 2) + 0.0:,.2f}, fair spread {cds_result.fair_spread:.4%}")

## Next

The [documentation](https://nanoql.org/docs/) has the guides, the API reference and the bindings' rules.